# AICL - Adaptive In-Context Learning Example

Full pipeline: retrieval > AICL > reader

Based on Chandra et al., ECIR 2025.

In [ ]:
import pandas as pd
import numpy as np
from pyterrier_rag.aicl import AICLContextSelector

In [ ]:
train_retrieved = pd.DataFrame({"qid":["q1","q1","q1","q2","q2","q2","q3","q3","q3"],"query":["what is python"]*3+["who is einstein"]*3+["what is java"]*3,"docno":["d1","d2","d3","d4","d5","d6","d7","d8","d9"],"score":[0.9,0.7,0.5,0.8,0.6,0.3,0.7,0.5,0.2],"rank":[0,1,2,0,1,2,0,1,2],"text":["Python is a programming language.","Python used for data science.","Python by Guido van Rossum.","Einstein was a physicist.","Einstein born in Germany.","Einstein relativity theory.","Java is object-oriented.","Java runs on JVM.","Java by Sun Microsystems."]})
print(train_retrieved)

In [ ]:
answers_df = pd.DataFrame({"qid":["q1","q2","q3"],"answers":["python","einstein","java"]})
train_labels = AICLContextSelector.build_labels_from_results(train_retrieved,answers_df,answer_col="answers",k_max=3)
for qid,label in zip(["q1","q2","q3"],train_labels):
    print(f"{qid}: {label}")

In [ ]:
aicl = AICLContextSelector(k_max=3)
aicl.fit(train_retrieved,train_labels)
print("AICL fitted.")

In [ ]:
test_retrieved = pd.DataFrame({"qid":["q4","q4","q4"],"query":["what is numpy"]*3,"docno":["d10","d11","d12"],"score":[0.85,0.60,0.30],"rank":[0,1,2],"text":["NumPy is a Python library.","NumPy provides arrays.","NumPy used in data science."]})
filtered = aicl.transform(test_retrieved)
print(f"Before AICL: {len(test_retrieved)} rows")
print(f"After AICL: {len(filtered)} rows")
print(filtered[["qid","docno","score","text"]])

## Real PyTerrier Pipeline
```python
bm25 = pt.BatchRetrieve(index, wmodel="BM25")
aicl = AICLContextSelector(k_max=20)
aicl.fit(train_retrieved, train_labels)
pipeline = bm25 % 20 >> aicl >> reader
results = pipeline(topics)
```